In [2]:
"""
=========================================================
MODELOS — VALIDACIÓN POR VENTANA / SEGMENTO (StratifiedKFold)
=========================================================
Versión "por segmento", equivalente metodológicamente a la
que usa Fraiwan et al. (2021): los folds se reparten de forma
aleatoria estratificada a nivel de VENTANA, sin agrupar por
paciente. Es la validación COMPARABLE con la literatura.

Un único script para las TRES configuraciones:
  - ICBHI sola
  - Fraiwan sola
  - Fusión (ICBHI + Fraiwan)

(El script gemelo con StratifiedGroupKFold da la validación
 por paciente, más estricta, para el análisis crítico.)

CÓMO USARLO:
  Cambia CONFIG a "ICBHI", "FRAIWAN" o "FUSION".
=========================================================
"""

import re
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from collections import Counter

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    cohen_kappa_score, balanced_accuracy_score
)
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

# =========================================================
# >>> PARÁMETROS A ELEGIR <<<
# =========================================================

CONFIG   = "FUSION"   # "ICBHI", "FRAIWAN" o "FUSION"
N_SPLITS = 10

# =========================================================
# RUTAS
# =========================================================

path_icbhi   = r"C:\Users\josem\Desktop\tfg\features_enriquecidas_corregido.csv"
path_fraiwan = r"C:\Users\josem\Desktop\tfg\features_fraiwan_limpio.csv"

# =========================================================
# CARGA
# =========================================================

def cargar(path):
    return pd.read_csv(path, sep=';', decimal=',')

if CONFIG == "ICBHI":
    df = cargar(path_icbhi)
elif CONFIG == "FRAIWAN":
    df = cargar(path_fraiwan)
elif CONFIG == "FUSION":
    df_i = cargar(path_icbhi)
    df_f = cargar(path_fraiwan)
    if list(df_i.columns) != list(df_f.columns):
        raise ValueError("Las columnas de ICBHI y Fraiwan no coinciden.")
    df = pd.concat([df_i, df_f], ignore_index=True)
else:
    raise ValueError("CONFIG debe ser 'ICBHI', 'FRAIWAN' o 'FUSION'")

print("=" * 55)
print(f"CONFIGURACIÓN: {CONFIG}  —  VALIDACIÓN POR VENTANA  (N_SPLITS = {N_SPLITS})")
print("=" * 55)

# =========================================================
# LIMPIEZA
# =========================================================

df = df.dropna(subset=['archivo'])
df = df[df['archivo'].astype(str).str.strip() != '']

df['diagnostico'] = (
    df['diagnostico']
    .astype(str).str.strip().str.upper()
    .replace({'HEALTHY': 'NORMAL', 'BRONCHIECTASIS': 'BRON'})
)

classes_to_keep = ['COPD', 'NORMAL', 'BRON', 'PNEUMONIA', 'ASTHMA', 'HEART FAILURE']
df = df[df['diagnostico'].isin(classes_to_keep)]

print("\nDistribución por ventanas:")
print(df['diagnostico'].value_counts())

meta_cols    = ['archivo', 'diagnostico', 'ventana']
# (puede haber una columna patient_id residual; la excluimos si está)
meta_cols   += [c for c in ['patient_id', 'patient_id_raw', 'etiqueta_cruda'] if c in df.columns]
feature_cols = [c for c in df.columns if c not in meta_cols]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=feature_cols)

X = df[feature_cols].values
y = df['diagnostico'].values

print(f"\nFeatures: {len(feature_cols)}")
print(f"Total ventanas (tras dropna): {len(df)}")

# =========================================================
# EVALUACIÓN
# =========================================================

def evaluar(nombre, y_true, y_pred):
    acc     = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    kappa   = cohen_kappa_score(y_true, y_pred)
    labels  = sorted(set(y_true))
    cm      = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df   = pd.DataFrame(cm, index=labels, columns=labels)

    print(f"\n{'='*55}")
    print(f"  {nombre}")
    print(f"{'='*55}")
    print(f"  Accuracy:          {acc:.4f}")
    print(f"  Balanced Accuracy: {bal_acc:.4f}")
    print(f"  Cohen's Kappa:     {kappa:.4f}")
    print(f"\n{classification_report(y_true, y_pred, zero_division=0)}")
    print("Matriz de confusión:")
    print(cm_df)
    return bal_acc

# =========================================================
# CV POR VENTANA CON SMOTE DENTRO DEL FOLD
# =========================================================

def cross_val_ventana(nombre, modelo_fn, X, y, n_splits):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    all_true, all_pred = [], []

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        sc = MinMaxScaler()
        X_tr  = sc.fit_transform(X_tr)
        X_val = sc.transform(X_val)

        counter     = Counter(y_tr)
        min_samples = min(counter.values())
        k_neighbors = min(5, min_samples - 1)

        if k_neighbors < 1:
            X_tr_res, y_tr_res = X_tr, y_tr
        else:
            smt = SMOTETomek(
                smote=SMOTE(k_neighbors=k_neighbors, random_state=42),
                random_state=42)
            X_tr_res, y_tr_res = smt.fit_resample(X_tr, y_tr)

        modelo = modelo_fn()
        modelo.fit(X_tr_res, y_tr_res)
        y_pred = modelo.predict(X_val)

        bal = balanced_accuracy_score(y_val, y_pred)
        scores.append(bal)
        all_true.extend(y_val)
        all_pred.extend(y_pred)

        print(f"    Fold {fold+1}: Balanced Acc = {bal:.4f}")

    scores = np.array(scores)
    print(f"\n  Media:  {scores.mean():.4f}")
    print(f"  Std:    {scores.std():.4f}")

    evaluar(f"{nombre} — CV agregada", all_true, all_pred)
    return scores

# =========================================================
# MODELOS
# =========================================================

def make_adaboost():
    return AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=4, min_samples_leaf=2,
            class_weight='balanced', random_state=42),
        n_estimators=200, learning_rate=0.5, random_state=42)

def make_bagging():
    return BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=None, class_weight='balanced', random_state=42),
        n_estimators=200, random_state=42, n_jobs=-1)

def make_rf():
    return RandomForestClassifier(
        n_estimators=500, class_weight='balanced_subsample',
        max_features='sqrt', random_state=42, n_jobs=-1)

def make_svm():
    return SVC(kernel='rbf', C=10, gamma='scale',
               class_weight='balanced', random_state=42)

# =========================================================
# EJECUCIÓN
# =========================================================

for nombre, fn in [("ADABOOST", make_adaboost),
                   ("BAGGING", make_bagging),
                   ("RANDOM FOREST", make_rf),
                   ("SVM RBF", make_svm)]:
    print(f"\n{'='*40}")
    print(f"{nombre} — {N_SPLITS}-fold CV POR VENTANA con SMOTE")
    print(f"{'='*40}")
    cross_val_ventana(nombre, fn, X, y, N_SPLITS)

# =========================================================
# MODELO FINAL
# =========================================================

print(f"\n{'='*40}")
print("MODELO FINAL (entrenado en todo el dataset)")
print(f"{'='*40}")

scaler_final = MinMaxScaler()
X_scaled_all = scaler_final.fit_transform(X)

counter_all = Counter(y)
k_final     = min(5, min(counter_all.values()) - 1)
if k_final < 1:
    X_final_res, y_final_res = X_scaled_all, y
else:
    smt_final = SMOTETomek(
        smote=SMOTE(k_neighbors=k_final, random_state=42),
        random_state=42)
    X_final_res, y_final_res = smt_final.fit_resample(X_scaled_all, y)

modelo_final = make_rf()
modelo_final.fit(X_final_res, y_final_res)

print(f"Distribución tras SMOTE: {Counter(y_final_res)}")

importances = pd.Series(modelo_final.feature_importances_, index=feature_cols)
top = importances.sort_values(ascending=False).head(15)
print("\nTop 15 features:")
for feat, imp in top.items():
    bar = '█' * int(imp * 200)
    print(f"  {feat:<30} {imp:.4f}  {bar}")

print(f"\n{'='*40}")
print(f"FIN — CONFIGURACIÓN {CONFIG} (POR VENTANA)")
print(f"{'='*40}")

CONFIGURACIÓN: FUSION  —  VALIDACIÓN POR VENTANA  (N_SPLITS = 10)

Distribución por ventanas:
diagnostico
COPD             3474
NORMAL            448
ASTHMA            283
PNEUMONIA         199
HEART FAILURE     168
BRON              137
Name: count, dtype: int64

Features: 41
Total ventanas (tras dropna): 4708

ADABOOST — 10-fold CV POR VENTANA con SMOTE
    Fold 1: Balanced Acc = 0.6736
    Fold 2: Balanced Acc = 0.6585
    Fold 3: Balanced Acc = 0.5863
    Fold 4: Balanced Acc = 0.6403
    Fold 5: Balanced Acc = 0.6511
    Fold 6: Balanced Acc = 0.7488
    Fold 7: Balanced Acc = 0.6508
    Fold 8: Balanced Acc = 0.7098
    Fold 9: Balanced Acc = 0.6293
    Fold 10: Balanced Acc = 0.6939

  Media:  0.6643
  Std:    0.0429

  ADABOOST — CV agregada
  Accuracy:          0.8534
  Balanced Accuracy: 0.6646
  Cohen's Kappa:     0.6821

               precision    recall  f1-score   support

       ASTHMA       0.53      0.59      0.56       283
         BRON       0.79      0.70      0.74